<a href="https://colab.research.google.com/github/sig-gis/cwcb-landcover-mapping/blob/main/00_Intial_Explorations/sam3_batch_classify_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hard-coded concept classification with SAM 3, batch over a folder

The concept sets are written down, not discovered. Each class carries its definition, the types
under it, and the synonyms under each type. Every synonym is a text prompt; every detection keeps
its term, type and class, so results roll up at any of the three levels.

Run order: Cells 1 to 5 set the model and the concept input up once. Cell 6 declares the run scope
and where results land. Cells 7 and 8 define the per-scene work. Cell 9 runs every scene found.

Every scene in the sample folder is processed. Anything under a `nir` folder is skipped. Each
scene lands in its own folder on Drive holding four things: `instances.gpkg` (one polygon per
detection carrying class, type, term and score), `classes.tif` (those polygons burned to a class
raster, highest score winning any overlap), `detections.csv` (the same attributes flat, no
geometry), and `summary.json` (the scene's parameters, per class and type and term counts with
score statistics, the prompts that detected nothing, and timings). A run-level `manifest.csv`
records every scene's status.

Nothing scales with scene size in RAM. Tiles are cut to local disk and dropped when the scene
finishes, results are assembled per scene and copied to Drive, the class raster is burned one
block at a time, and both display reads are decimated. A scene that fails is logged and the run
continues to the next one. A finished scene is skipped on a re-run, so an interrupted run resumes
where it stopped.

Prompts are batched against one tile encoding: the vision encoder runs once per tile and the
embeddings are expanded across a batch of prompts, so a tile costs `ceil(len(PROMPTS)/PROMPT_BATCH)`
forwards rather than `len(PROMPTS)`.

In [9]:
# ── Cell 1: install ──
# Leaves torch alone, so a local ROCm build is untouched.
!pip -q install -U transformers accelerate huggingface_hub rasterio geopandas ipywidgets

In [10]:
# ── Cell 2: imports, device, AOI discovery ──
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]      = "expandable_segments:True"

import gc, colorsys, json, shutil, time, traceback
import numpy as np
import torch
import rasterio
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from rasterio.windows import Window
from rasterio.features import shapes as rio_shapes, rasterize
from shapely.geometry import shape as shp_shape
from shapely.ops import unary_union
from PIL import Image
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# The source scenes and nothing else. digitization_samples_nir is a sibling of this folder, so
# pointing one level down leaves NIR out by structure, with no filtering to get wrong.
DRIVE_AOI_DIR  = "/content/drive/MyDrive/digitization_samples_with_imagery-1/digitization_samples"
DRIVE_OUT_ROOT = "/content/drive/MyDrive/sam3_classified"   # results. Never an input.
WORK_DIR       = "/content/sam3_work"                       # scratch, local disk

OUTPUTS = ["instances.gpkg", "classes.tif", "detections.csv", "summary.json", "overlay.png"]

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("drive mount skipped:", e)

def discover_aois(base):
    """Every GeoTIFF under `base`, labelled by its path relative to base."""
    base = Path(base)
    if not base.exists():
        root = Path("/content/drive/MyDrive")
        hits = list(root.rglob(base.name)) if root.exists() else []
        base = hits[0] if hits else base
    tifs = sorted(list(base.rglob("*.tif")) + list(base.rglob("*.tiff")))
    out = []
    for t in tifs:
        try: label = str(t.relative_to(base))
        except ValueError: label = t.name
        out.append((label, str(t)))
    return str(base), out

def is_source_scene(label):
    """A classification is not a scene. Existing ones sit beside their source as
    <scene>_textclasses.tif; this notebook writes classes.tif and cuts tile_*.tif."""
    n = Path(label).name.lower()
    return not (n.endswith("_textclasses.tif") or n == "classes.tif" or n.startswith("tile_"))

def scenes_in(base):
    """Discovery unchanged, minus any GeoTIFF that is a classification rather than a scene."""
    aoi_base, found = discover_aois(base)
    keep = [(l, p) for l, p in found if is_source_scene(l)]
    drop = [Path(l).name for l, p in found if not is_source_scene(l)]
    if drop:
        print(f"ignored {len(drop)} classified raster(s): "
              f"{', '.join(drop[:3])}{' ...' if len(drop) > 3 else ''}")
    return aoi_base, keep

AOI_BASE, AOIS = scenes_in(DRIVE_AOI_DIR)
print(f"AOI dir: {AOI_BASE}  (exists: {Path(AOI_BASE).exists()})")
print(f"{len(AOIS)} scene(s) to classify")
for label, _ in AOIS: print("   ", label)

def out_key(scene_path):
    """Where a scene's results go, mirroring its place under the AOI folder. Two sample folders
    can hold the same filename, so the stem alone would collide and one scene would be lost."""
    p = Path(scene_path)
    try: rel = p.relative_to(AOI_BASE)
    except ValueError: rel = Path(p.name)
    return rel.parent / rel.stem

_keys = [str(out_key(p)) for _, p in AOIS]
assert len(_keys) == len(set(_keys)), "two scenes resolve to one output folder"

if not AOIS:                                  # say what IS there, so DRIVE_AOI_DIR can be fixed
    root = Path("/content/drive/MyDrive")
    print(f"\nnothing found. {root} exists: {root.exists()}")
    if root.exists():
        print("folders in MyDrive:")
        for d in sorted(p for p in root.iterdir() if p.is_dir())[:40]:
            print("   ", d.name)
    print("\nset DRIVE_AOI_DIR above to the folder holding the GeoTIFFs and re-run this cell.")

device: cuda
Mounted at /content/drive
ignored 4 classified raster(s): 2025-05-12_us-co-denver-2025_sample_four_textclasses.tif, 2025-05-12_us-co-denver-2025_sample_one_textclasses.tif, 2025-05-02_us-co-boulder-2025_sample_one_textclasses.tif ...
AOI dir: /content/drive/MyDrive/digitization_samples_with_imagery-1/digitization_samples  (exists: True)
9 scene(s) to classify
    Castle Rock/Sample Four/2025-05-12_us-co-denver-2025_sample_four.tif
    Castle Rock/Sample One/2025-05-12_us-co-denver-2025_sample_one.tif
    Castle Rock/Sample Three/2025-05-12_us-co-denver-2025_sample_three.tif
    Castle Rock/Sample Two/2025-05-12_us-co-denver-2025_sample_two.tif
    Westminster/Sample Four/2025-05-02_us-co-boulder-2025_sample_four.tif
    Westminster/Sample Four/2025-05-12_us-co-denver-2025_sample_four.tif
    Westminster/Sample One/2025-05-02_us-co-boulder-2025_sample_one.tif
    Westminster/Sample Three/2025-05-02_us-co-boulder-2025_sample_three.tif
    Westminster/Sample Two/2025-05-02_us

In [11]:
# ── Cell 3: authenticate (SAM 3 is gated) ──
from huggingface_hub import login

tok = os.environ.get("HF_TOKEN")
if tok is None:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        tok = None

if tok:
    login(tok); print("logged in")
else:
    login()

logged in


In [12]:
# ── Cell 4: load SAM 3 ──
from transformers import Sam3Model, Sam3Processor

dtype = torch.bfloat16 if device == "cuda" else torch.float32
model = Sam3Model.from_pretrained("facebook/sam3", dtype=dtype).to(device).eval()
processor = Sam3Processor.from_pretrained("facebook/sam3")

MODEL_PX = processor.image_processor.size["height"]   # whatever the processor stretches input to
print("loaded | model input:", MODEL_PX, "px")

loaded | model input: 1008 px


In [13]:
# ── Cell 5: THE INPUT. Edit this cell and nothing else to change the task. ──
# class -> (irrigation status, definition, {type: [synonyms]}).
# The synonyms are hand written. The definition is documentation for the reader; the leaves under
# it are what SAM 3 is actually prompted with. Every detection keeps its term, type and class.
#
# Prompt count drives runtime directly. Cutting a type to a single best term is the cheapest
# speedup available and costs nothing but recall on that term's phrasing.

CLASSES = {
 "structures": ("NI",
   "Houses, garages, sheds, decks, swing sets, and solar panels on a structure",
   {"house":        ["house", "rooftop", "residential building"],
    "outbuilding":  ["garage", "shed"],
    "deck":         ["deck", "patio"],
    "roof solar":   ["solar panel on a roof"]}),

 "roads": ("NI",
   "Roads: gravel and asphalt",
   {"paved road":   ["road", "asphalt road", "street"],
    "unpaved road": ["gravel road", "dirt road"],
    "driveway":     ["driveway"]}),

 "concrete": ("NI",
   "Concrete, pavers, and brick",
   {"concrete":     ["concrete", "concrete slab"],
    "paving":       ["pavers", "brick paving", "sidewalk"]}),

 "impervious other": ("NI",
   "Other impervious objects: solar panels on the ground, tarps over pools, shade tarps, "
   "retention walls, fences, boulders, rocks, tennis courts, or movable objects presumably on "
   "impervious surfaces such as garbage bins, umbrellas, patio furniture, and trampolines",
   {"ground solar": ["solar panel array on the ground"],
    "cover":        ["tarp", "pool cover", "shade sail"],
    "barrier":      ["retaining wall", "fence"],
    "rock":         ["boulder", "rock"],
    "court":        ["tennis court"],
    "yard object":  ["garbage bin", "patio umbrella", "patio furniture"]}),

 "artificial turf": ("NI",
   "Artificial turf",
   {"artificial turf": ["artificial turf", "astroturf", "synthetic grass"]}),

 "pools": ("II",
   "Pools (inset or above ground), hot tubs, koi ponds, and man-made water features or "
   "fountains larger than 64 square feet",
   {"pool":         ["swimming pool"],
    "spa":          ["hot tub"],
    "water feature":["koi pond", "fountain"]}),

 "turf": ("II, INI",
   "Manicured lawns, smooth in texture",
   {"lawn":         ["lawn", "mowed grass", "turf grass"]}),

 "canopy": ("II, INI, NI",
   "Shrubs, trees, and other vegetation large enough to cast shadows",
   {"tree":         ["tree", "tree canopy"],
    "shrub":        ["shrub", "bush", "hedge"]}),

 "ground cover": ("II, INI, NI",
   "Ground cover such as landscaping mulch and rock, or coarse grasses that do not cast shadows",
   {"mulch":        ["landscaping mulch", "wood chips"],
    "rock cover":   ["decorative gravel", "landscaping rock"],
    "coarse grass": ["coarse grass", "ornamental grass"]}),

 "bare earth": ("II, INI, NI",
   "Bare earth lacking vegetation. Must be between irrigated plantings or orchards to be "
   "considered irrigated",
   {"bare earth":   ["bare soil", "bare dirt", "bare ground"]}),

 "vehicles": ("II, INI, NI",
   "Vehicles, tractors, or other movable objects such as garbage bins and umbrellas that retain "
   "the irrigation status of the land they are on",
   {"road vehicle": ["car", "truck"],
    "equipment":    ["tractor", "trailer"]}),

 "trampolines": ("II, INI, NI",
   "Trampolines that retain the irrigation status of the land they are on",
   {"trampoline":   ["trampoline"]}),

 "undeveloped lands": ("NI",
   "Areas deemed not irrigated by humans. Abandoned urban lots or native landscapes containing "
   "trees, grasses, and wetlands",
   {"vacant lot":   ["vacant lot"],
    "native cover": ["wild grass", "scrubland"],
    "wetland":      ["wetland", "marsh"]}),

 "horse corrals": ("NI",
   "Horse corrals and arenas, usually round or oval. Generally smooth soil texture, may contain "
   "signs of watering",
   {"corral":       ["horse corral", "paddock"],
    "arena":        ["riding arena"]}),

 "open water": ("NI",
   "Ocean coastline, lakes, rivers, or retention ponds",
   {"still water":  ["lake", "pond"],
    "moving water": ["river", "ocean"]}),

 "agricultural lands": ("NI",
   "Large commercial agriculture identified by vegetation planted in rows (row crops, vineyards, "
   "nurseries), trees planted in formations or rows (fruit and nut orchards or nurseries), clear "
   "signs of management with irrigation or intention to irrigate (plowed, tilled, circular "
   "irrigation patterns, flood irrigation, presence of pivots or irrigation lines or discolored "
   "soils), or irrigated livestock pastureland",
   {"row crop":     ["row crop field", "plowed field"],
    "orchard":      ["orchard", "vineyard"],
    "nursery":      ["plant nursery"],
    "pasture":      ["pasture"]}),
}

# Flatten to the prompt list. PROMPTS[i] is the text SAM 3 sees; PROMPT_OF[i] is where it belongs.
PROMPTS, PROMPT_OF = [], []
for cname, (irr, defn, types) in CLASSES.items():
    for tname, terms in types.items():
        for term in terms:
            PROMPTS.append(term); PROMPT_OF.append((cname, tname, term))

CLASS_NAMES = list(CLASSES)
CLASS_IDX   = {c: i for i, c in enumerate(CLASS_NAMES)}

for cname, (irr, defn, types) in CLASSES.items():
    n = sum(len(v) for v in types.values())
    print(f"{cname:20s} {irr:>12s}  {n:2d} terms in {len(types)} type(s)")
print(f"\n{len(CLASSES)} classes, {len(PROMPTS)} prompts")

structures                     NI   8 terms in 4 type(s)
roads                          NI   6 terms in 3 type(s)
concrete                       NI   5 terms in 2 type(s)
impervious other               NI  12 terms in 6 type(s)
artificial turf                NI   3 terms in 1 type(s)
pools                          II   4 terms in 3 type(s)
turf                      II, INI   3 terms in 1 type(s)
canopy                II, INI, NI   5 terms in 2 type(s)
ground cover          II, INI, NI   6 terms in 3 type(s)
bare earth            II, INI, NI   3 terms in 1 type(s)
vehicles              II, INI, NI   4 terms in 2 type(s)
trampolines           II, INI, NI   1 terms in 1 type(s)
undeveloped lands              NI   5 terms in 3 type(s)
horse corrals                  NI   3 terms in 2 type(s)
open water                     NI   4 terms in 2 type(s)
agricultural lands             NI   6 terms in 4 type(s)

16 classes, 78 prompts


In [14]:
# ── Cell 6: the run scope, and where results land ──
# Every scene Cell 2 found is processed. The scan re-runs here, so a folder that gained a file
# since Cell 2 is picked up, and this run's own results are never mistaken for scenes.
#
# A scene's results are assembled on local disk and copied to DRIVE_OUT_ROOT/<scene stem>/ once
# complete, so a half-written result never reaches Drive and a killed run leaves nothing to
# untangle. A scene whose folder already holds every output is skipped unless OVERWRITE, which is
# what makes an interrupted run resumable.

OVERWRITE    = False   # True re-does scenes that already carry a complete result on Drive
KEEP_TILES   = False   # True leaves a scene's cut tiles on local disk after it finishes
SHOW_OVERLAY = True    # draw each scene's overlay inline as it finishes. The PNG is written either way

AOI_BASE, AOIS = scenes_in(DRIVE_AOI_DIR)    # re-scan, so re-running this cell refreshes
assert AOIS, (f"no GeoTIFF under {AOI_BASE}. Fix DRIVE_AOI_DIR in Cell 2, re-run it, "
              f"then re-run this cell.")

SCENES = [p for _, p in AOIS]
Path(DRIVE_OUT_ROOT).mkdir(parents=True, exist_ok=True)
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

def scene_done(scene_path):
    """A scene counts as done when summary.json landed. It is copied last."""
    return (Path(DRIVE_OUT_ROOT) / out_key(scene_path) / "summary.json").exists()

_done = [s for s in SCENES if scene_done(s)]
print(f"{len(SCENES)} scene(s) in scope under {AOI_BASE}")
print(f"results -> {DRIVE_OUT_ROOT}")
print(f"{len(_done)} already complete, {len(SCENES) - len(_done)} to run"
      f"{' (OVERWRITE is on, all will re-run)' if OVERWRITE else ''}\n")
for label, p in AOIS:
    print(f"  [{'done' if scene_done(p) else '    '}] {label}")
print(f"\n{len(SCENES)} scenes x {len(PROMPTS)} prompts. Run Cells 7 and 8 to define the work, "
      f"then Cell 9 to do it.")

ignored 4 classified raster(s): 2025-05-12_us-co-denver-2025_sample_four_textclasses.tif, 2025-05-12_us-co-denver-2025_sample_one_textclasses.tif, 2025-05-02_us-co-boulder-2025_sample_one_textclasses.tif ...
9 scene(s) in scope under /content/drive/MyDrive/digitization_samples_with_imagery-1/digitization_samples
results -> /content/drive/MyDrive/sam3_classified
9 already complete, 0 to run

  [done] Castle Rock/Sample Four/2025-05-12_us-co-denver-2025_sample_four.tif
  [done] Castle Rock/Sample One/2025-05-12_us-co-denver-2025_sample_one.tif
  [done] Castle Rock/Sample Three/2025-05-12_us-co-denver-2025_sample_three.tif
  [done] Castle Rock/Sample Two/2025-05-12_us-co-denver-2025_sample_two.tif
  [done] Westminster/Sample Four/2025-05-02_us-co-boulder-2025_sample_four.tif
  [done] Westminster/Sample Four/2025-05-12_us-co-denver-2025_sample_four.tif
  [done] Westminster/Sample One/2025-05-02_us-co-boulder-2025_sample_one.tif
  [done] Westminster/Sample Three/2025-05-02_us-co-boulder-202

In [15]:
# ── Cell 7: cut a scene into GeoTIFF tiles on disk ──
# Whole tiles only: SAM 3 stretches any input to MODEL_PX square, so a short edge tile would land
# at a different effective scale. The last row and column sit flush against the edge instead.
# Each tile carries its own transform, so a tile's pixels convert to real coordinates on their own
# and nothing downstream needs to know where the tile came from.
#
# Same cut as always, addressed by scene rather than by a picked one. Cell 9 calls this per scene.

TILE_PX  = 1008     # native px per tile. 1008 == MODEL_PX == 1:1, a native pixel is a model pixel.
OVERLAP  = 0.30     # fractional overlap between tiles. 0 is fastest and cuts edge objects.

def _origins(n, tile, overlap):
    """Whole tiles only; the final one is pushed flush to the edge."""
    t = min(tile, n)
    stride = max(1, int(t * (1 - overlap)))
    xs = list(range(0, max(1, n - t + 1), stride))
    if xs[-1] != n - t: xs.append(max(0, n - t))
    return xs, t

def tile_scene(scene_path):
    """Cut one scene to tiles. Returns (tiles, TW, TH, src_profile)."""
    tile_dir = Path(WORK_DIR) / out_key(scene_path) / "tiles"
    tile_dir.mkdir(parents=True, exist_ok=True)

    with rasterio.open(scene_path) as src:
        print(f"scene: {Path(scene_path).name} | {src.width} x {src.height} | {src.count} band(s) "
              f"| {src.dtypes[0]} | {src.crs}")
        xs, TW = _origins(src.width, TILE_PX, OVERLAP)
        ys, TH = _origins(src.height, TILE_PX, OVERLAP)

        lohi = None                                   # uint8 RGB passes through untouched. Anything
        if src.dtypes[0] != "uint8":                  # else gets one global 2-98% stretch, computed
            s = min(2048 / max(src.width, src.height), 1.0)          # once from a decimated read
            ov = src.read(out_shape=(src.count, max(1, int(src.height * s)),
                                     max(1, int(src.width * s))), out_dtype="float32")
            ov = ov[:3] if ov.shape[0] >= 3 else np.repeat(ov[:1], 3, 0)
            lohi = [np.percentile(ov[b], (2, 98)) for b in range(3)]
            del ov
            print(f"stretch (2-98% per band): {[[round(float(v), 1) for v in p] for p in lohi]}")

        tiles = []
        for oy in ys:
            for ox in xs:
                out = tile_dir / f"tile_{ox:06d}_{oy:06d}.tif"
                tiles.append(str(out))
                if out.exists(): continue                            # re-running is free
                w = Window(ox, oy, TW, TH)
                a = src.read(window=w, out_dtype="float32" if lohi else "uint8")
                a = a[:3] if a.shape[0] >= 3 else np.repeat(a[:1], 3, 0)
                if lohi:
                    a = np.stack([np.clip((a[b] - lohi[b][0]) / max(lohi[b][1] - lohi[b][0], 1e-6),
                                          0, 1) * 255 for b in range(3)]).astype(np.uint8)
                with rasterio.open(out, "w", driver="GTiff", height=TH, width=TW, count=3,
                                   dtype="uint8", crs=src.crs,
                                   transform=rasterio.windows.transform(w, src.transform),
                                   compress="deflate") as dst:
                    dst.write(a)
        src_profile = dict(height=src.height, width=src.width, crs=src.crs,
                           transform=src.transform)

    print(f"{len(tiles)} tile(s) of {TW} x {TH} px in {tile_dir}")
    return tiles, TW, TH, src_profile

In [16]:
# ── Cell 8: classify one scene, write instances, class raster and stats ──
# One vision encoding per tile, expanded across a batch of prompts, so a tile costs
# ceil(len(PROMPTS)/PROMPT_BATCH) forwards. PROMPT_BATCH=1 is the old one-prompt-per-forward
# behaviour. On CUDA OOM the batch halves and retries, down to 1, so a big batch cannot kill a run.
# The halved batch is held in BATCH_STATE and carries to the next scene: once a size is known to
# fit this GPU there is no reason to rediscover it a hundred times.
#
# Instances keep their identity: each mask becomes a polygon in the scene CRS carrying class, type,
# term and score. The class raster is burned from those polygons at the end, lowest score first, so
# the highest scoring instance wins any overlap.
#
# score is SAM 3's detection confidence for that prompt: sigmoid(pred_logits) * sigmoid(presence_
# logits), the object's own confidence times the tile-level confidence that the concept is present
# at all. It survives to every output: the polygon attribute, the CSV column, the class raster's
# burn order, and the per class, type and term statistics in summary.json.

PROMPT_BATCH = 8       # prompts per forward. Lower it if the run OOMs at every tile.
THRESH       = 0.35    # detection score cutoff
IOU_DEDUP    = 0.5     # box IoU above which two instances are the same object across a seam
DISPLAY      = 1400    # long-side of the shown overlay

BATCH_STATE = dict(batch=PROMPT_BATCH)     # survives scenes, only ever shrinks

def _expand(vis, n):
    """Reuse one tile's vision embeddings across n prompts. expand() is a view, not a copy."""
    return type(vis)(
        fpn_hidden_states=tuple(t.expand(n, *t.shape[1:]) for t in vis.fpn_hidden_states),
        fpn_position_encoding=tuple(t.expand(n, *t.shape[1:]) for t in vis.fpn_position_encoding))

def _amp(fn, **kw):
    with torch.no_grad():
        if device == "cuda":
            with torch.autocast("cuda", dtype=torch.bfloat16): return fn(**kw)
        return fn(**kw)

def _run_batch(vis, idx):
    """Forward one batch of prompt indices against a tile's embeddings -> list of results."""
    txt = processor(text=[PROMPTS[i] for i in idx], return_tensors="pt").to(device)  # padded to 32
    out = _amp(model, vision_embeds=_expand(vis, len(idx)),
               input_ids=txt["input_ids"], attention_mask=txt["attention_mask"])
    res = processor.post_process_instance_segmentation(
        out, threshold=THRESH, mask_threshold=0.5, target_sizes=None)
    packed = [(r["masks"].float().cpu().numpy(), r["scores"].float().cpu().numpy()) for r in res]
    del out, txt
    return packed

def _iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1]); ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    if ix0 >= ix1 or iy0 >= iy1: return 0.0
    inter = (ix1 - ix0) * (iy1 - iy0)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

def _palette(n):
    return [colorsys.hsv_to_rgb((i * 0.61803398875) % 1.0, 0.65, 0.95) for i in range(n)]
PALETTE = _palette(len(CLASS_NAMES))

def classify_scene(scene_path, tiles, TW, TH, src_profile, out_dir):
    """Detect, dedup, write instances + class raster + stats for one scene. Returns the summary."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    GPKG    = out_dir / "instances.gpkg"
    CLS_TIF = out_dir / "classes.tif"
    CSV     = out_dir / "detections.csv"
    PNG     = out_dir / "overlay.png"
    SUMMARY = out_dir / "summary.json"
    t0 = time.time()

    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

    rows = []
    for ti, tf in enumerate(tiles):
        with rasterio.open(tf) as t:
            arr, ttr = t.read([1, 2, 3]), t.transform
        win = Image.fromarray(arr.transpose(1, 2, 0)); del arr

        pv = processor(images=win, return_tensors="pt")["pixel_values"].to(device)
        vis = _amp(model.vision_encoder, pixel_values=pv); del pv       # encode once per tile

        i = 0
        while i < len(PROMPTS):
            idx = list(range(i, min(i + BATCH_STATE["batch"], len(PROMPTS))))
            try:
                packed = _run_batch(vis, idx)
            except torch.cuda.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                if BATCH_STATE["batch"] == 1: raise
                BATCH_STATE["batch"] = max(1, BATCH_STATE["batch"] // 2)
                print(f"\nOOM: PROMPT_BATCH -> {BATCH_STATE['batch']}")
                continue
            for pi, (lm, sc) in zip(idx, packed):
                cname, tname, term = PROMPT_OF[pi]
                for mi, si in zip(lm, sc):
                    m = np.asarray(Image.fromarray((mi > 0).astype(np.uint8) * 255)
                                   .resize((TW, TH), Image.NEAREST)).astype(np.uint8)
                    if not m.any(): continue
                    geoms = [shp_shape(g) for g, v in rio_shapes(m, mask=m.astype(bool),
                                                                 transform=ttr)]
                    if not geoms: continue
                    rows.append(dict(geometry=unary_union(geoms), cls=cname, type=tname,
                                     term=term, score=float(si), tile=Path(tf).name))
            i += len(idx)
        del vis
        if device == "cuda": torch.cuda.empty_cache()
        print(f"tile {ti + 1}/{len(tiles)}: {len(rows)} instance(s)", end="\r")

    n_raw = len(rows)
    print(f"\n{n_raw} raw instance(s)")

    if n_raw:
        gdf = gpd.GeoDataFrame(rows, crs=src_profile["crs"])
        gdf = gdf.sort_values("score", ascending=False).reset_index(drop=True)  # seam dedup on
        keep, grid = [], {}                                 # bounds, bucketed to stay linear
        cell = TILE_PX * abs(src_profile["transform"].a)
        for i, b in enumerate(gdf.geometry.bounds.itertuples(index=False)):
            gx, gy = int(b[0] // cell), int(b[1] // cell)
            near = [k for dx in (-1, 0, 1) for dy in (-1, 0, 1) for k in grid.get((gx+dx, gy+dy), ())]
            if any(_iou(b, k) >= IOU_DEDUP for k in near): continue
            grid.setdefault((gx, gy), []).append(b); keep.append(i)
        gdf = gdf.iloc[keep].reset_index(drop=True)
        gdf["cls_id"] = gdf["cls"].map(lambda c: CLASS_IDX[c] + 1).astype("uint8")
        if GPKG.exists(): GPKG.unlink()
        gdf.to_file(GPKG, layer="instances", driver="GPKG")
        print(f"{len(gdf)} instance(s) kept -> {GPKG}")
    else:
        gdf = gpd.GeoDataFrame({"geometry": [], "cls": [], "type": [], "term": [], "score": [],
                                "tile": [], "cls_id": []}, geometry="geometry",
                               crs=src_profile["crs"])
        print("no detections. writing an empty result for this scene")
    del rows

    cent = gdf.geometry.centroid if len(gdf) else None       # flat attribute table, no geometry
    flat = gdf.drop(columns="geometry").copy()
    flat["x"] = cent.x.values if len(gdf) else []
    flat["y"] = cent.y.values if len(gdf) else []
    flat.sort_values("score", ascending=False).to_csv(CSV, index=False)
    print(f"detections table -> {CSV}")

    prof = dict(driver="GTiff", height=src_profile["height"], width=src_profile["width"], count=1,
                dtype="uint8", crs=src_profile["crs"], transform=src_profile["transform"],
                nodata=0, compress="lzw", tiled=True, blockxsize=512, blockysize=512)
    sidx      = gdf.sindex if len(gdf) else None
    low_first = gdf.sort_values("score") if len(gdf) else gdf   # later burns over earlier
    with rasterio.open(CLS_TIF, "w", **prof) as dst:            # one block at a time, bounded RAM
        for _, w in dst.block_windows(1):
            if sidx is None:
                dst.write(np.zeros((w.height, w.width), "uint8"), 1, window=w); continue
            wtr = rasterio.windows.transform(w, src_profile["transform"])
            bb  = rasterio.windows.bounds(w, src_profile["transform"])
            hit = low_first.loc[low_first.index.intersection(sidx.query(
                gpd.GeoSeries.from_wkt([f"POLYGON(({bb[0]} {bb[1]},{bb[2]} {bb[1]},"
                                        f"{bb[2]} {bb[3]},{bb[0]} {bb[3]},{bb[0]} {bb[1]}))"])[0]))]
            a = (rasterize([(g, v) for g, v in zip(hit.geometry, hit.cls_id)],
                           out_shape=(w.height, w.width), transform=wtr, fill=0, dtype="uint8")
                 if len(hit) else np.zeros((w.height, w.width), "uint8"))
            dst.write(a, 1, window=w)
    print(f"class raster -> {CLS_TIF}")

    with rasterio.open(scene_path) as src:                   # both reads decimated, bounded RAM
        s = min(DISPLAY / max(src.width, src.height), 1.0)
        w, h = max(1, int(src.width * s)), max(1, int(src.height * s))
        base = src.read(out_shape=(src.count, h, w), out_dtype="float32")[:3]
        base = np.moveaxis(base, 0, -1) / 255.0
    with rasterio.open(CLS_TIF) as c:
        small = c.read(1, out_shape=(h, w), resampling=rasterio.enums.Resampling.nearest)

    ov = base.copy()
    for i, cname in enumerate(CLASS_NAMES):
        sel = small == i + 1
        if sel.any(): ov[sel] = 0.40 * base[sel] + 0.60 * np.array(PALETTE[i])

    share = gdf.groupby("cls").size() if len(gdf) else {}
    fig = plt.figure(figsize=(14, 10)); plt.imshow(np.clip(ov, 0, 1)); plt.axis("off")
    handles = [Patch(facecolor=PALETTE[i], label=f"{c} ({int(share.get(c, 0))})")
               for i, c in enumerate(CLASS_NAMES) if len(gdf) and share.get(c, 0)]
    if handles: plt.legend(handles=handles, loc="upper right", framealpha=0.9)
    plt.title(f"{Path(scene_path).name}\n{len(gdf)} instance(s) across {len(tiles)} tiles "
              f"@ {TILE_PX}px  (score >= {THRESH})")
    plt.savefig(PNG, dpi=110, bbox_inches="tight")
    if SHOW_OVERLAY: plt.show()
    plt.close(fig); del ov, base, small          # figures and arrays go now, not at GC's leisure

    for cname in CLASS_NAMES:                                  # class > type > term
        sub = gdf[gdf["cls"] == cname]
        if not len(sub): continue
        print(f"{cname}  {len(sub)} instance(s)")
        for tname, g in sub.groupby("type"):
            print(f"    {tname:14s} " + "  ".join(f"{t}: {n}" for t, n in g["term"].value_counts()
                                                  .items()))
    dead = sorted(set(PROMPTS) - set(gdf["term"]))
    print(f"\n{len(dead)} prompt(s) detected nothing: {dead}")

    def _stats(g):
        """The confidence roll-up, kept at every level results roll up at."""
        return dict(n=int(len(g)), score_min=round(float(g["score"].min()), 4),
                    score_mean=round(float(g["score"].mean()), 4),
                    score_max=round(float(g["score"].max()), 4))

    by_class = {}
    for cname in CLASS_NAMES:
        sub = gdf[gdf["cls"] == cname]
        if not len(sub): continue
        by_class[cname] = dict(irrigation=CLASSES[cname][0], **_stats(sub),
            types={tn: dict(**_stats(g),
                            terms={tm: _stats(gg) for tm, gg in g.groupby("term")})
                   for tn, g in sub.groupby("type")})

    summary = dict(
        scene=Path(scene_path).name, scene_path=str(scene_path),
        width=src_profile["width"], height=src_profile["height"],
        crs=str(src_profile["crs"]), transform=list(src_profile["transform"])[:6],
        tiles=len(tiles), tile_px=TILE_PX, tile_w=TW, tile_h=TH, overlap=OVERLAP,
        prompts=len(PROMPTS), classes=len(CLASSES),
        prompt_batch=BATCH_STATE["batch"], thresh=THRESH, iou_dedup=IOU_DEDUP,
        instances_raw=n_raw, instances_kept=int(len(gdf)),
        by_class=by_class, dead_prompts=dead,
        seconds=round(time.time() - t0, 1),
        outputs=list(OUTPUTS))
    SUMMARY.write_text(json.dumps(summary, indent=2))
    print(f"summary -> {SUMMARY}")

    del gdf, flat
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    return summary

In [17]:
# ── Cell 9: run every scene ──
# Each scene is tiled, classified, then published to Drive as a unit. A scene that raises is
# recorded and the run moves on, so one bad file cannot end a hundred good ones. Tiles are dropped
# and the allocator is released between scenes, which is what keeps the last scene as affordable as
# the first. summary.json is copied last, so a scene only reads as done when everything before it
# arrived.

matplotlib.rcParams["figure.max_open_warning"] = 0
MANIFEST = Path(DRIVE_OUT_ROOT) / "manifest.csv"

def publish(local_dir, drive_dir):
    """Copy a finished scene to Drive. summary.json goes last: it is the done marker."""
    drive_dir.mkdir(parents=True, exist_ok=True)
    for name in [n for n in OUTPUTS if n != "summary.json"] + ["summary.json"]:
        src = Path(local_dir) / name
        if src.exists(): shutil.copy2(src, drive_dir / name)
    return drive_dir

manifest, t_run = [], time.time()
for si, scene in enumerate(SCENES, 1):
    key       = out_key(scene)
    drive_dir = Path(DRIVE_OUT_ROOT) / key
    local_dir = Path(WORK_DIR) / key
    head      = f"[{si}/{len(SCENES)}] {Path(scene).name}"

    if scene_done(scene) and not OVERWRITE:
        print(f"{head}: done already, skipping")
        manifest.append(dict(scene=Path(scene).name, status="skipped", tiles=0, instances_raw=0,
                             instances_kept=0, seconds=0.0, out=str(drive_dir), error=""))
        continue

    print(f"\n{'=' * 78}\n{head}\n{'=' * 78}")
    t0, row = time.time(), dict(scene=Path(scene).name, status="failed", tiles=0, instances_raw=0,
                                instances_kept=0, seconds=0.0, out=str(drive_dir), error="")
    try:
        tiles, TW, TH, prof = tile_scene(scene)
        summary = classify_scene(scene, tiles, TW, TH, prof, local_dir)
        publish(local_dir, drive_dir)
        row.update(status="ok", tiles=summary["tiles"], instances_raw=summary["instances_raw"],
                   instances_kept=summary["instances_kept"])
        print(f"{head}: published -> {drive_dir}")
    except KeyboardInterrupt:
        print(f"\n{head}: interrupted. {si - 1} scene(s) finished, re-run this cell to resume")
        raise
    except Exception as e:
        row["error"] = f"{type(e).__name__}: {e}"[:300]
        print(f"\n{head}: FAILED, moving on\n{traceback.format_exc()}")
    finally:
        row["seconds"] = round(time.time() - t0, 1)
        manifest.append(row)
        if not KEEP_TILES: shutil.rmtree(local_dir / "tiles", ignore_errors=True)
        plt.close("all"); gc.collect()
        if device == "cuda": torch.cuda.empty_cache()

import pandas as pd
mf = pd.DataFrame(manifest)
mf.to_csv(MANIFEST, index=False)

ok   = int((mf["status"] == "ok").sum())
skip = int((mf["status"] == "skipped").sum())
bad  = mf[mf["status"] == "failed"]
print(f"\n{'=' * 78}")
print(f"{ok} ok, {skip} skipped, {len(bad)} failed of {len(SCENES)} scene(s) "
      f"in {round((time.time() - t_run) / 60, 1)} min")
print(f"{int(mf['instances_kept'].sum())} instance(s) kept across the run")
for _, r in bad.iterrows(): print(f"  failed: {r['scene']}  {r['error']}")
print(f"\nper scene: {DRIVE_OUT_ROOT}/<scene>/  ({', '.join(OUTPUTS)})")
print(f"manifest:  {MANIFEST}")
print("class raster: 0 = unlabelled, else CLASS_NAMES index + 1")
mf

[1/9] 2025-05-12_us-co-denver-2025_sample_four.tif: done already, skipping
[2/9] 2025-05-12_us-co-denver-2025_sample_one.tif: done already, skipping
[3/9] 2025-05-12_us-co-denver-2025_sample_three.tif: done already, skipping
[4/9] 2025-05-12_us-co-denver-2025_sample_two.tif: done already, skipping
[5/9] 2025-05-02_us-co-boulder-2025_sample_four.tif: done already, skipping
[6/9] 2025-05-12_us-co-denver-2025_sample_four.tif: done already, skipping
[7/9] 2025-05-02_us-co-boulder-2025_sample_one.tif: done already, skipping
[8/9] 2025-05-02_us-co-boulder-2025_sample_three.tif: done already, skipping
[9/9] 2025-05-02_us-co-boulder-2025_sample_two.tif: done already, skipping

0 ok, 9 skipped, 0 failed of 9 scene(s) in 0.0 min
0 instance(s) kept across the run

per scene: /content/drive/MyDrive/sam3_classified/<scene>/  (instances.gpkg, classes.tif, detections.csv, summary.json, overlay.png)
manifest:  /content/drive/MyDrive/sam3_classified/manifest.csv
class raster: 0 = unlabelled, else CLASS

,scene,status,tiles,instances_raw,instances_kept,seconds,out,error
0,2025-05-12_us-co-denver-2025_sample_four.tif,skipped,0,0,0,0.0,/content/drive/MyDrive/sam3_classified/Castle ...,
1,2025-05-12_us-co-denver-2025_sample_one.tif,skipped,0,0,0,0.0,/content/drive/MyDrive/sam3_classified/Castle ...,
2,2025-05-12_us-co-denver-2025_sample_three.tif,skipped,0,0,0,0.0,/content/drive/MyDrive/sam3_classified/Castle ...,
3,2025-05-12_us-co-denver-2025_sample_two.tif,skipped,0,0,0,0.0,/content/drive/MyDrive/sam3_classified/Castle ...,
4,2025-05-02_us-co-boulder-2025_sample_four.tif,skipped,0,0,0,0.0,/content/drive/MyDrive/sam3_classified/Westmin...,
5,2025-05-12_us-co-denver-2025_sample_four.tif,skipped,0,0,0,0.0,/content/drive/MyDrive/sam3_classified/Westmin...,
6,2025-05-02_us-co-boulder-2025_sample_one.tif,skipped,0,0,0,0.0,/content/drive/MyDrive/sam3_classified/Westmin...,
7,2025-05-02_us-co-boulder-2025_sample_three.tif,skipped,0,0,0,0.0,/content/drive/MyDrive/sam3_classified/Westmin...,
8,2025-05-02_us-co-boulder-2025_sample_two.tif,skipped,0,0,0,0.0,/content/drive/MyDrive/sam3_classified/Westmin...,


In [18]:
# ── Cell 10: review objects and a plot fishnet for Collect Earth Online ──
# The instance layer overlaps on purpose. Several classes are overlays that keep the irrigation
# status of the ground beneath them, and for the rest the class raster already resolved overlap by
# letting the highest score win. Digitizers correct objects rather than pixels, so this cell cuts
# each scene into review objects, hands each one a proposed class read back from classes.tif and the
# instance that best explains it, and flags the ones a person should look at. Because every object
# carries exactly one class, the layer a digitizer edits never overlaps itself even though the
# instances behind it do.
#
# The objects are one continuous surface across the whole scene. SLIC runs once over a decimated
# read of the entire scene, so an object boundary never snaps to a tile or block edge and no
# seamlines appear between tiles. The labels are polygonised on the decimated grid, and each
# object's class make-up is read from classes.tif at full resolution, streamed block by block, so
# nothing scales with scene size in RAM beyond the one decimated image SLIC needs. Raise SP_MAXDIM
# for finer object edges at more memory, lower it for less.
#
# Two tiers go to CEO. A 25 m fishnet is the plot: PLOTID counts 1..M over the cells that hold at
# least one object. Each object is a sample: SAMPLEID is the object's own id, and PLOTID is the
# fishnet cell its representative point lands in, the plot it belongs to. Both tiers export as
# singlepart-polygon GeoJSON in EPSG:4326, which is what CEO reads. The GeoPackage stays as the
# QGIS review layer and now carries plot_id and sample_id so the two views agree.
#
# It runs after Cell 9, reading each scene's published classes.tif and instances.gpkg from Drive.
# One scene at a time. A scene already carrying samples.geojson is skipped unless SP_OVERWRITE. A
# scene that fails is logged and the run moves on.

!pip -q install -U scikit-image >/dev/null

import pandas as pd
from rasterio import Affine
from shapely.geometry import box
from skimage.segmentation import slic

SP_REGION_PX = 28      # target object size in native pixels. Smaller makes more, finer objects
SP_COMPACT   = 12.0    # SLIC compactness. Higher is squarer objects, lower hugs image edges harder
SP_MAXDIM    = 5000    # scene is decimated so its long side is at most this before one global SLIC
SP_STREAM    = 2048    # classes.tif is read back in blocks of this many px, bounded RAM
SP_PURITY    = 0.80    # a review object whose dominant class holds less than this share is flagged
PLOT_SIZE_M  = 25.0    # side of the CEO plot fishnet, in metres of the scene's projected CRS
SP_OVERWRITE = False   # True re-makes objects for scenes that already carry them

SP_LAYER, SP_NAME        = "superpixels", "superpixels.gpkg"
PLOTS_NAME, SAMPLES_NAME = "plots.geojson", "samples.geojson"
ID2CLS = {i + 1: c for i, c in enumerate(CLASS_NAMES)}     # class raster value -> class name

def _sp_decimate(src):
    """A whole-scene RGB read decimated so the long side is <= SP_MAXDIM, HxWx3 float in 0-1, given
    the same 2-98% per-band stretch Cell 7 renders a non-uint8 scene with. Returns the image and the
    decimated grid's transform."""
    s = min(SP_MAXDIM / max(src.width, src.height), 1.0)
    Wd, Hd = max(1, int(round(src.width * s))), max(1, int(round(src.height * s)))
    a = src.read(out_shape=(src.count, Hd, Wd),
                 out_dtype="float32" if src.dtypes[0] != "uint8" else "uint8")
    a = a[:3] if a.shape[0] >= 3 else np.repeat(a[:1], 3, 0)
    if src.dtypes[0] != "uint8":
        lohi = [np.percentile(a[b], (2, 98)) for b in range(3)]
        a = np.stack([np.clip((a[b] - lo) / max(hi - lo, 1e-6), 0, 1)
                      for b, (lo, hi) in enumerate(lohi)])
        img = np.moveaxis(a, 0, -1)
    else:
        img = np.moveaxis(a, 0, -1).astype(np.float32) / 255.0
    dec_t = src.transform * Affine.scale(src.width / Wd, src.height / Hd)
    return img, dec_t

def _sp_classhist(cls, lab, nc):
    """Per-object class make-up as a labels x raster-ids table. classes.tif is read in blocks and
    each native pixel is mapped to its decimated label, so only one block and the label array are
    ever in RAM."""
    Hd, Wd = lab.shape
    H, W   = cls.height, cls.width
    nlab   = int(lab.max())
    hist   = np.zeros((nlab + 1) * nc, dtype=np.int64)
    for oy in range(0, H, SP_STREAM):
        rr = (np.arange(oy, min(oy + SP_STREAM, H)) * Hd // H)
        for ox in range(0, W, SP_STREAM):
            cc  = (np.arange(ox, min(ox + SP_STREAM, W)) * Wd // W)
            win = Window(ox, oy, len(cc), len(rr))
            cl  = cls.read(1, window=win).astype(np.int64)
            lb  = lab[np.ix_(rr, cc)].astype(np.int64)
            hist += np.bincount(lb.ravel() * nc + cl.ravel(), minlength=hist.size)
            del cl, lb
    return hist.reshape(nlab + 1, nc)

def review_objects_for_scene(scene_path, out_dir):
    """Partition one scene into a continuous object surface, label each object from classes.tif and
    instances.gpkg, assign each to a 25 m plot, and write the QGIS GeoPackage and the CEO GeoJSON
    pair."""
    out_dir = Path(out_dir)
    cls_tif, gpkg_in = out_dir / "classes.tif", out_dir / "instances.gpkg"
    assert cls_tif.exists(), f"no classes.tif in {out_dir}, run Cell 9 for this scene first"

    polys, recs = [], []
    with rasterio.open(scene_path) as src, rasterio.open(cls_tif) as cls:
        scene_crs   = src.crs
        img, dec_t  = _sp_decimate(src)
        n_seg = max(1, int((src.width * src.height) / (SP_REGION_PX ** 2)))
        try:
            lab = slic(img, n_segments=n_seg, compactness=SP_COMPACT, start_label=1, channel_axis=-1)
        except TypeError:                                          # older scikit-image
            lab = slic(img, n_segments=n_seg, compactness=SP_COMPACT, start_label=1, multichannel=True)
        del img
        nc   = len(CLASS_NAMES) + 1
        hist = _sp_classhist(cls, lab, nc)                         # rows are labels, cols raster ids

        next_id = 1
        for g, v in rio_shapes(lab.astype(np.int32), transform=dec_t):
            k = int(v)
            if k == 0:                                             # start_label=1, so 0 is none
                continue
            counts = hist[k]; tot = int(counts.sum())
            if tot == 0:
                continue
            dom = int(counts.argmax())
            polys.append(shp_shape(g))
            recs.append(dict(sp_id=next_id, class_auto=ID2CLS.get(dom, "unlabelled"),
                             purity=round(float(counts[dom] / tot), 3),
                             n_class=int((counts[1:] > 0).sum()), px=tot))
            next_id += 1
        del lab, hist
        gc.collect()

    sp = gpd.GeoDataFrame(recs, geometry=polys, crs=scene_crs)
    sp["area"] = sp.geometry.area

    # reconcile with the instances: the highest-scoring one touching each object, and how many
    # distinct classes reach into it at all
    for c, dv in dict(sam_cls=None, sam_term=None, sam_type=None, sam_score=np.nan,
                      n_sam_cls=0).items():
        sp[c] = dv
    if gpkg_in.exists():
        inst = gpd.read_file(gpkg_in, layer="instances")
        if len(inst):
            inst = inst.to_crs(sp.crs)
            j = gpd.sjoin(sp[["sp_id", "geometry"]],
                          inst[["cls", "type", "term", "score", "geometry"]],
                          predicate="intersects", how="inner")
            if len(j):
                top  = j.sort_values("score", ascending=False).drop_duplicates("sp_id") \
                        .set_index("sp_id")
                ncls = j.groupby("sp_id")["cls"].nunique()
                sp = sp.set_index("sp_id")
                sp.loc[top.index, "sam_cls"]   = top["cls"]
                sp.loc[top.index, "sam_term"]  = top["term"]
                sp.loc[top.index, "sam_type"]  = top["type"]
                sp.loc[top.index, "sam_score"] = top["score"].round(4)
                sp.loc[ncls.index, "n_sam_cls"] = ncls.astype(int)
                sp = sp.reset_index()

    # one flag, three reasons: the object is mixed, more than one class claims it, or the raster and
    # the best instance name different classes for it
    mixed    = sp["purity"] < SP_PURITY
    competed = sp["n_sam_cls"] > 1
    disagree = sp["sam_cls"].notna() & (sp["sam_cls"] != sp["class_auto"])
    sp["conflict"] = (mixed | competed | disagree).astype(int)

    # the 25 m plot fishnet. Each object's representative point picks the cell it lands in; only
    # cells that catch an object become plots, renumbered PLOTID 1..M. sample_id is the object's own
    # id, so a sample keeps its identity while pointing at the plot it belongs to.
    minx, miny, maxx, maxy = sp.total_bounds
    rp  = sp.geometry.representative_point()
    col = np.floor((rp.x.values - minx) / PLOT_SIZE_M).astype(np.int64)
    row = np.floor((maxy - rp.y.values) / PLOT_SIZE_M).astype(np.int64)
    order, plot_id = {}, np.empty(len(sp), dtype=np.int64)
    for i, rc in enumerate(zip(row.tolist(), col.tolist())):
        if rc not in order:
            order[rc] = len(order) + 1
        plot_id[i] = order[rc]
    sp["plot_id"], sp["sample_id"] = plot_id, sp["sp_id"]

    plot_polys, plot_recs = [], []
    for (r, c), pid in order.items():
        x0, y0 = minx + c * PLOT_SIZE_M, maxy - r * PLOT_SIZE_M
        plot_polys.append(box(x0, y0 - PLOT_SIZE_M, x0 + PLOT_SIZE_M, y0))
        plot_recs.append(dict(PLOTID=int(pid)))
    plots = gpd.GeoDataFrame(plot_recs, geometry=plot_polys, crs=scene_crs)
    plots["n_samples"] = plots["PLOTID"].map(pd.Series(plot_id).value_counts()).fillna(0).astype(int)

    # the correction surface a digitizer edits, prefilled with the automatic call
    sp["class_final"], sp["reviewed"], sp["notes"] = sp["class_auto"], 0, ""
    sp = sp[["sp_id", "plot_id", "sample_id", "class_auto", "class_final", "reviewed", "conflict",
             "purity", "n_class", "n_sam_cls", "sam_cls", "sam_term", "sam_type", "sam_score",
             "px", "area", "notes", "geometry"]]

    out_gpkg = out_dir / SP_NAME
    if out_gpkg.exists(): out_gpkg.unlink()
    sp.to_file(out_gpkg, layer=SP_LAYER, driver="GPKG")

    # the CEO pair: singlepart-polygon GeoJSON in EPSG:4326, the CRS CEO reads. Samples carry PLOTID
    # and SAMPLEID and the class fields a reviewer needs; plots carry PLOTID and their sample count.
    samples = sp.rename(columns=dict(plot_id="PLOTID", sample_id="SAMPLEID", class_auto="CLASS_AUTO",
                                     class_final="CLASS_FIN", reviewed="REVIEWED", conflict="CONFLICT",
                                     purity="PURITY", sam_cls="SAM_CLS", sam_score="SAM_SCORE"))
    samples = samples[["PLOTID", "SAMPLEID", "CLASS_AUTO", "CLASS_FIN", "REVIEWED", "CONFLICT",
                       "PURITY", "SAM_CLS", "SAM_SCORE", "geometry"]].to_crs(4326)
    plots_out = plots[["PLOTID", "n_samples", "geometry"]].to_crs(4326)
    for p, gdf in ((out_dir / PLOTS_NAME, plots_out), (out_dir / SAMPLES_NAME, samples)):
        if p.exists(): p.unlink()
        gdf.to_file(p, driver="GeoJSON")

    return dict(scene=Path(scene_path).name, objects=int(len(sp)), plots=int(len(plots)),
                flagged=int(sp["conflict"].sum()), gpkg=str(out_gpkg))

# run over every scene that already carries a result on Drive, the same discovery Cell 11 bundles
out_root = Path(DRIVE_OUT_ROOT)
assert out_root.exists() and any(out_root.rglob("summary.json")), \
    f"no results under {out_root}. Run Cell 9 first."

done = sorted(out_root.rglob("summary.json"))
print(f"{len(done)} scene(s) with results under {out_root}\n")
sp_rows = []
for si, sm in enumerate(done, 1):
    d, head = sm.parent, f"[{si}/{len(done)}] {sm.parent.name}"
    scene_path = json.loads(sm.read_text()).get("scene_path", "")
    if (d / SAMPLES_NAME).exists() and not SP_OVERWRITE:
        print(f"{head}: objects already present, skipping"); continue
    if not scene_path or not Path(scene_path).exists():
        print(f"{head}: scene image not found at {scene_path}, skipping"); continue
    try:
        r = review_objects_for_scene(scene_path, d)
        print(f"{head}: {r['objects']} objects in {r['plots']} plots, {r['flagged']} flagged "
              f"-> {r['gpkg']}")
        sp_rows.append(dict(status="ok", **r))
    except Exception as e:
        print(f"{head}: FAILED {type(e).__name__}: {e}")
        sp_rows.append(dict(scene=d.name, status="failed", objects=0, plots=0, flagged=0, gpkg="",
                            error=f"{type(e).__name__}: {e}"[:200]))
    finally:
        gc.collect()

if sp_rows:
    spm = pd.DataFrame(sp_rows); spm.to_csv(out_root / "superpixels_manifest.csv", index=False)
    ok = int((spm["status"] == "ok").sum())
    flagged = int(spm[spm["status"] == "ok"]["flagged"].sum()) if ok else 0
    print(f"\n{ok} scene(s) got objects, {flagged} object(s) flagged for review across the run")
    print(f"per scene: {DRIVE_OUT_ROOT}/<scene>/{SAMPLES_NAME} + {PLOTS_NAME}  (CEO), "
          f"{SP_NAME}  (QGIS)")
    print("load plots.geojson as CEO plots and samples.geojson as its samples, joined on PLOTID")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cucim-cu12 26.2.0 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.
9 scene(s) with results under /content/drive/MyDrive/sam3_classified



[1/9] 2025-05-12_us-co-denver-2025_sample_four: 14876 objects in 117 plots, 6585 flagged -> /content/drive/MyDrive/sam3_classified/Castle Rock/Sample Four/2025-05-12_us-co-denver-2025_sample_four/superpixels.gpkg
[2/9] 2025-05-12_us-co-denver-2025_sample_one: 57062 objects in 440 plots, 28213 flagged -> /content/drive/MyDrive/sam3_classified/Castle Rock/Sample One/2025-05-12_us-co-denver-2025_sample_one/superpixels.gpkg
[3/9] 2025-05-12_us-co-denver-2025_sample_three: 34574 objects in 270 plots, 22320 flagged -> /content/drive/MyDrive/sam3_classified/Castle Rock/Sample Three/2025-05-12_us-co-denver-2025_sample_three/superpixels.gpkg
[4/9] 2025-05-12_us-co-denver-2025_sample_two: 42765 objects in 336 plots, 25701 flagged -> /content/drive/MyDrive/sam3_classified/Castle Rock/Sample Two/2025-05-12_us-co-denver-2025_sample_two/superpixels.gpkg
[5/9] 2025-05-02_us-co-boulder-2025_sample_four: 20842 objects in 156 plots, 11526 flagged -> /content/drive/MyDrive/sam3_classified/Westminster/Sam

In [19]:
# ── Cell 11: bundle the run and take it off Colab ──
# Cell 9 published each scene to Drive as it finished, so the results are already there, spread
# across one folder per scene. This packs the whole run into a single archive, drops it on Drive
# beside those folders, and hands the same file to the browser.
#
# It reads what is on Drive rather than what this session ran, so it bundles every scene that ever
# completed. Safe to re-run on its own, and safe to run in a fresh session after a resumed run.

ZIP_STEM = f"sam3_classified_{time.strftime('%Y%m%d_%H%M')}"
DOWNLOAD = True     # False writes the archive to Drive and skips the browser download

src = Path(DRIVE_OUT_ROOT)
assert src.exists() and any(src.rglob("summary.json")), f"no results under {src}. Run Cell 9 first."

scenes  = sorted(str(p.parent.relative_to(src)) for p in src.rglob("summary.json"))
n_files = sum(1 for p in src.rglob("*") if p.is_file())
print(f"bundling {len(scenes)} scene(s), {n_files} file(s), from {src}")
for x in scenes: print("   ", x)

# Staged on local disk first. Zipping onto the Drive mount is slow, and a run killed partway
# through would leave a half-written archive sitting where a complete one is expected.
staged  = shutil.make_archive(f"/content/{ZIP_STEM}", "zip", root_dir=str(src))
size_mb = Path(staged).stat().st_size / 1e6

drive_zip = src.parent / f"{ZIP_STEM}.zip"     # beside sam3_classified, never inside it
if Path(staged).resolve() != drive_zip.resolve():   # a no-op if staging already landed there
    shutil.copy2(staged, drive_zip)
print(f"\narchive -> {drive_zip}  ({size_mb:.1f} MB)")

if DOWNLOAD:
    if size_mb > 500:
        print(f"{size_mb:.0f} MB is a lot for a browser download and it may time out. The Drive "
              f"copy is complete either way.")
    try:
        from google.colab import files
        files.download(staged)
    except Exception as e:
        print(f"browser download unavailable ({e}). Take it from Drive: {drive_zip}")

bundling 9 scene(s), 128 file(s), from /content/drive/MyDrive/sam3_classified
    Castle Rock/Sample Four/2025-05-12_us-co-denver-2025_sample_four
    Castle Rock/Sample One/2025-05-12_us-co-denver-2025_sample_one
    Castle Rock/Sample Three/2025-05-12_us-co-denver-2025_sample_three
    Castle Rock/Sample Two/2025-05-12_us-co-denver-2025_sample_two
    Westminster/Sample Four/2025-05-02_us-co-boulder-2025_sample_four
    Westminster/Sample Four/2025-05-12_us-co-denver-2025_sample_four
    Westminster/Sample One/2025-05-02_us-co-boulder-2025_sample_one
    Westminster/Sample Three/2025-05-02_us-co-boulder-2025_sample_three
    Westminster/Sample Two/2025-05-02_us-co-boulder-2025_sample_two

archive -> /content/drive/MyDrive/sam3_classified_20260806_2212.zip  (426.3 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>